# LeetCode 2418 — Sort the People bằng Python

Link đề: https://leetcode.com/problems/sort-the-people/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def sortPeople(self, names: List[str], heights: List[int]) -> List[str]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **sắp xếp hai mảng song song**: `names[i]` đi cùng `heights[i]`, sắp theo chiều cao **giảm dần** nhưng chỉ trả về tên.
2. Dùng `zip`, `sorted(..., key=..., reverse=True)` và cách **sắp xếp chỉ số** (`argsort`) để giữ cặp (tên, chiều cao) đi cùng nhau.
3. Hiểu vì sao có thể dùng `dict` chiều cao → tên (chiều cao **đôi một khác nhau**) nhưng không thể dùng `dict` tên → chiều cao (tên có thể trùng).
4. Viết 4 cách: sắp xếp chọn (selection sort) thủ công, `zip` + `sorted`, sắp xếp chỉ số, `dict` + chiều cao giảm dần; đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: sắp xếp tăng dần thay vì giảm dần, sắp theo **tên** thay vì chiều cao, mất tên trùng vì dùng `dict` sai chiều.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `names` và mảng số nguyên **đôi một khác nhau** `heights`, cả hai có cùng độ dài `n`. Người thứ `i` có tên `names[i]` và chiều cao `heights[i]`.

Hãy trả về mảng `names` sau khi **sắp xếp theo chiều cao giảm dần** (người cao nhất đứng đầu).

| Ví dụ | names | heights | Output |
|---|---|---|---|
| 1 | `["Mary","John","Emma"]` | `[180,165,170]` | `["Mary","Emma","John"]` |
| 2 | `["Alice","Bob","Bob"]` | `[155,185,150]` | `["Bob","Alice","Bob"]` |

Giải thích ví dụ 1: Mary cao 180 (cao nhất), Emma 170, John 165. Ví dụ 2: Bob (185), Alice (155), Bob (150) — hai người tên `Bob` là **hai người khác nhau** với chiều cao khác nhau.

Constraints: `n == names.length == heights.length`; `1 <= n <= 10^3`; `1 <= names[i].length <= 20`; `1 <= heights[i] <= 10^5`; `names[i]` chỉ gồm chữ cái tiếng Anh thường và hoa; mọi giá trị trong `heights` **khác nhau**.

In [1]:
names = ["Mary", "John", "Emma"]
heights = [180, 165, 170]
for i in range(len(names)):
    print(f"người {i}: {names[i]:5s} cao {heights[i]}")
print("n =", len(names), "| heights khác nhau?", len(set(heights)) == len(heights))

người 0: Mary  cao 180
người 1: John  cao 165
người 2: Emma  cao 170
n = 3 | heights khác nhau? True


## 3. Kiến thức nền

### 3.1. Hai mảng song song và `zip`

`zip(a, b)` ghép phần tử cùng chỉ số thành các cặp. Ở đây cặp `(heights[i], names[i])` mô tả đầy đủ một người. Kết quả của `zip` là iterator; dùng `list(...)` để xem.

In [2]:
names = ["Mary", "John", "Emma"]
heights = [180, 165, 170]
print(list(zip(heights, names)))
print(list(zip(names, heights)))

[(180, 'Mary'), (165, 'John'), (170, 'Emma')]
[('Mary', 180), ('John', 165), ('Emma', 170)]


### 3.2. `sorted`, `key` và `reverse`

`sorted(iterable)` trả list mới. Mặc định sắp **tăng dần**; `reverse=True` đảo thành giảm dần. Khi sắp xếp các **tuple**, Python so sánh phần tử đầu tiên, nếu bằng mới xét phần tử kế tiếp. Vì vậy thứ tự các thành phần trong cặp rất quan trọng!

In [3]:
pairs = [(180, "Mary"), (165, "John"), (170, "Emma")]
print(sorted(pairs))                    # tăng dần theo chiều cao
print(sorted(pairs, reverse=True))      # giảm dần theo chiều cao
swapped = [("Mary", 180), ("John", 165), ("Emma", 170)]
print(sorted(swapped, reverse=True))    # sắp theo TÊN (sai ý đề)

[(165, 'John'), (170, 'Emma'), (180, 'Mary')]
[(180, 'Mary'), (170, 'Emma'), (165, 'John')]
[('Mary', 180), ('John', 165), ('Emma', 170)]


### 3.3. Tham số `key` — "sắp theo cái gì"

`key=` nhận một hàm, được gọi cho từng phần tử để lấy **khoá so sánh**. `sorted(pairs, key=lambda p: p[1])` sắp theo phần tử thứ hai của mỗi cặp.

In [4]:
words = ["pear", "fig", "banana"]
print(sorted(words, key=len))
print(sorted(words, key=len, reverse=True))
print(sorted(pairs, key=lambda p: -p[0]))

['fig', 'pear', 'banana']
['banana', 'pear', 'fig']
[(180, 'Mary'), (170, 'Emma'), (165, 'John')]


### 3.4. Sắp xếp chỉ số (argsort)

Thay vì sắp xếp chính dữ liệu, ta sắp xếp danh sách **chỉ số** `0..n-1` theo khoá lấy từ `heights`: `sorted(range(n), key=lambda i: heights[i], reverse=True)`. Kết quả là thứ tự chỉ số; sau đó dựng `[names[i] for i in order]`.

In [5]:
order = sorted(range(len(heights)), key=lambda i: heights[i], reverse=True)
print("thứ tự chỉ số:", order)
print([names[i] for i in order])

thứ tự chỉ số: [0, 2, 1]
['Mary', 'Emma', 'John']


### 3.5. `dict`: khoá phải **duy nhất**

`dict` chỉ giữ **một** giá trị cho mỗi khoá; khoá trùng sẽ ghi đè. Vì `heights` đôi một khác nhau nên `dict(zip(heights, names))` an toàn, còn `dict(zip(names, heights))` có thể **mất người** khi tên trùng.

In [6]:
names2, heights2 = ["Alice", "Bob", "Bob"], [155, 185, 150]
print(dict(zip(heights2, names2)))      # 3 mục, đủ
print(dict(zip(names2, heights2)))      # chỉ 2 mục: Bob bị ghi đè

{155: 'Alice', 185: 'Bob', 150: 'Bob'}
{'Alice': 155, 'Bob': 150}


### 3.6. `max` với `key` và `list.pop`

`max(range(n), key=lambda i: h[i])` cho chỉ số của phần tử lớn nhất. `lst.pop(i)` xoá phần tử ở vị trí `i` và trả về nó (tốn `O(n)`). Hai công cụ này đủ để tự cài **selection sort** — Cách 1.

In [7]:
h = [180, 165, 170]
i = max(range(len(h)), key=lambda t: h[t])
print("chỉ số cao nhất:", i, h[i])
x = h.pop(i)
print("pop ->", x, "còn", h)

chỉ số cao nhất: 0 180
pop -> 180 còn [165, 170]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy xếp hàng một lớp học theo chiều cao từ cao đến thấp. Mỗi bạn có **một tấm thẻ tên**. Bạn không cần sắp riêng "chiều cao" rồi "tên"; chỉ cần **cặp** (chiều cao, tên) đi cùng nhau và sắp theo **chiều cao**.

Ví dụ 1: các cặp `(180, Mary)`, `(165, John)`, `(170, Emma)`.

```
Bước 1: sắp theo chiều cao giảm dần: (180, Mary) (170, Emma) (165, John)
Bước 2: chỉ giữ lại tên:               Mary        Emma        John
```

Đáp án `["Mary", "Emma", "John"]` khớp với đề.

Một cách khác để nghĩ: vì chiều cao **đôi một khác nhau**, chiều cao là một "mã số duy nhất" của mỗi người. Nghĩa là ta có thể tra cứu: chiều cao nào → tên nào. Sắp xếp các chiều cao giảm dần rồi tra tên tương ứng.

## 5. Cách 1 — Brute Force / baseline

**Selection sort thủ công:** lặp `n` lần; mỗi lần tìm người **cao nhất còn lại** (trong số chưa xếp), thêm tên vào kết quả rồi đánh dấu/loại bỏ.

Cách này không dùng hàm `sorted` nên giúp ta hiểu bản chất "tìm max lặp đi lặp lại". Nút thắt: mỗi lần tìm max tốn `O(n)`, `n` lần → `O(n^2)`.

In [8]:
class SolutionBrute:
    def sortPeople(self, names, heights):
        names = list(names)          # sao chép để không phá dữ liệu gốc
        heights = list(heights)
        result = []
        while names:
            best = 0
            for i in range(1, len(heights)):
                if heights[i] > heights[best]:
                    best = i
            result.append(names.pop(best))
            heights.pop(best)
        return result

print(SolutionBrute().sortPeople(["Mary", "John", "Emma"], [180, 165, 170]))
print(SolutionBrute().sortPeople(["Alice", "Bob", "Bob"], [155, 185, 150]))

['Mary', 'Emma', 'John']
['Bob', 'Alice', 'Bob']


**Độ phức tạp:** vòng ngoài `n` lần, mỗi lần quét `O(n)` và `pop` `O(n)` → Time `O(n^2)`. Với `n <= 1000` vẫn chạy được (~10^6), nhưng sai tinh thần "dùng sort có sẵn". Space `O(n)` cho bản sao + kết quả.

## 6. Ý tưởng thuật toán / bước đột phá

Python đã có thuật toán sắp xếp `O(n log n)` (Timsort). Việc của ta chỉ là **đưa cho nó đúng khoá**:

- Ghép `(heights[i], names[i])` thành cặp, sắp theo phần tử đầu (chiều cao) giảm dần. Do chiều cao khác nhau, **không bao giờ cần xét tên để phá hòa**.
- Hoặc sắp xếp chỉ số theo `heights`, rồi tra `names`.

**Quy tắc "nếu… thì…":**

- Nếu cặp được dựng là `(chiều cao, tên)` và dùng `reverse=True` → chiều cao quyết định thứ tự.
- Nếu dựng là `(tên, chiều cao)` → sai: tên quyết định thứ tự.
- Nếu chiều cao có thể trùng → cần thêm khoá phụ (ở bài này không xảy ra).

Invariant: sau khi sắp xếp, dãy chiều cao là giảm ngặt, và mỗi tên vẫn đi cùng chiều cao ban đầu của mình.

In [9]:
names = ["Mary", "John", "Emma"]
heights = [180, 165, 170]
paired = sorted(zip(heights, names), reverse=True)
print("cặp đã sắp xếp:", paired)
print("chỉ lấy tên    :", [name for _, name in paired])

cặp đã sắp xếp: [(180, 'Mary'), (170, 'Emma'), (165, 'John')]
chỉ lấy tên    : ['Mary', 'Emma', 'John']


## 7. Các cách tiếp theo

Ta có 4 cách: (1) selection sort thủ công, (2) `zip` + `sorted`, (3) sắp xếp chỉ số, (4) `dict` chiều cao → tên.

### Cách 2 — `zip` + `sorted`

Dựng cặp `(chiều cao, tên)`, sắp giảm dần, lấy tên. Ngắn gọn, đọc lên như đề bài.

In [10]:
class SolutionZip:
    def sortPeople(self, names, heights):
        return [name for _, name in sorted(zip(heights, names), reverse=True)]

print(SolutionZip().sortPeople(["Mary", "John", "Emma"], [180, 165, 170]))
print(SolutionZip().sortPeople(["Alice", "Bob", "Bob"], [155, 185, 150]))

['Mary', 'Emma', 'John']
['Bob', 'Alice', 'Bob']


Độ phức tạp Cách 2: Time `O(n log n)` (sắp xếp), Space `O(n)`. Lưu ý: với tên trùng và chiều cao khác nhau vẫn đúng vì chiều cao quyết định; chỉ khi chiều cao trùng thì Python mới so sánh tên.

### Cách 3 — Sắp xếp chỉ số (argsort)

Không tạo cặp; sắp `range(n)` theo khoá `heights[i]` giảm dần. Rất hữu ích khi có **nhiều mảng song song** (tên, tuổi, điểm…) vì chỉ cần một `order` để sắp lại tất cả.

In [11]:
class SolutionIndex:
    def sortPeople(self, names, heights):
        order = sorted(range(len(names)), key=lambda i: heights[i], reverse=True)
        return [names[i] for i in order]

print(SolutionIndex().sortPeople(["Mary", "John", "Emma"], [180, 165, 170]))
print(SolutionIndex().sortPeople(["Alice", "Bob", "Bob"], [155, 185, 150]))

['Mary', 'Emma', 'John']
['Bob', 'Alice', 'Bob']


Độ phức tạp Cách 3: Time `O(n log n)`, Space `O(n)` (list chỉ số + kết quả).

### Cách 4 — `dict` chiều cao → tên

Vì chiều cao đôi một khác nhau, ta lập `who = dict(zip(heights, names))`, sắp các **chiều cao** giảm dần rồi tra tên. Đây là cách "bảng tra cứu" — dùng được nhờ đảm bảo của đề (khoá duy nhất).

In [12]:
class SolutionDict:
    def sortPeople(self, names, heights):
        who = dict(zip(heights, names))
        return [who[h] for h in sorted(heights, reverse=True)]

print(SolutionDict().sortPeople(["Mary", "John", "Emma"], [180, 165, 170]))
print(SolutionDict().sortPeople(["Alice", "Bob", "Bob"], [155, 185, 150]))

['Mary', 'Emma', 'John']
['Bob', 'Alice', 'Bob']


Độ phức tạp Cách 4: Time `O(n log n)` (sắp `heights`) cộng `O(n)` tra bảng, Space `O(n)`. Điều kiện: chiều cao phải **khác nhau** (đúng theo constraints).

## 8. TRACE từng bước

Trace Cách 1 (selection sort) với `names = ["Alice","Bob","Bob"]`, `heights = [155,185,150]`.

| vòng | `names` còn | `heights` còn | chỉ số max | thêm vào kết quả |
|---|---|---|---|---|
| 1 | `[Alice, Bob, Bob]` | `[155, 185, 150]` | 1 (185) | `Bob` |
| 2 | `[Alice, Bob]` | `[155, 150]` | 0 (155) | `Alice` |
| 3 | `[Bob]` | `[150]` | 0 (150) | `Bob` |

In [13]:
def trace_selection(names, heights):
    names, heights = list(names), list(heights)
    result = []
    round_no = 0
    while names:
        round_no += 1
        best = 0
        for i in range(1, len(heights)):
            if heights[i] > heights[best]:
                best = i
        print(f"vòng {round_no}: names={names} heights={heights} -> chọn chỉ số {best} ({names[best]}, {heights[best]})")
        result.append(names.pop(best))
        heights.pop(best)
    print("kết quả:", result)
    return result

trace_selection(["Alice", "Bob", "Bob"], [155, 185, 150])

vòng 1: names=['Alice', 'Bob', 'Bob'] heights=[155, 185, 150] -> chọn chỉ số 1 (Bob, 185)
vòng 2: names=['Alice', 'Bob'] heights=[155, 150] -> chọn chỉ số 0 (Alice, 155)
vòng 3: names=['Bob'] heights=[150] -> chọn chỉ số 0 (Bob, 150)
kết quả: ['Bob', 'Alice', 'Bob']


['Bob', 'Alice', 'Bob']

## 9. Code nộp LeetCode

In [14]:
from typing import List

class Solution:
    def sortPeople(self, names: List[str], heights: List[int]) -> List[str]:
        order = sorted(range(len(names)), key=lambda i: heights[i], reverse=True)
        return [names[i] for i in order]

print(Solution().sortPeople(["Mary", "John", "Emma"], [180, 165, 170]))
print(Solution().sortPeople(["Alice", "Bob", "Bob"], [155, 185, 150]))

['Mary', 'Emma', 'John']
['Bob', 'Alice', 'Bob']


## 10. Vì sao thuật toán đúng

- **Giữ nguyên cặp:** sắp xếp chỉ số (hoặc cặp) không bao giờ tách `names[i]` khỏi `heights[i]`; ta chỉ đổi **thứ tự** các chỉ số, rồi đọc tên theo thứ tự đó.
- **Thứ tự đúng:** `sorted(..., reverse=True)` theo khoá `heights[i]` cho dãy khoá không tăng; vì chiều cao đôi một khác nhau nên dãy này **giảm ngặt**, tức không có hai người "hoà nhau" và kết quả là duy nhất.
- **Tên trùng không sao:** thuật toán thao tác trên chỉ số/chiều cao, không dùng tên làm khoá; hai người cùng tên vẫn là hai chỉ số khác nhau.
- **Selection sort đúng:** mỗi vòng lấy phần tử lớn nhất còn lại; bằng quy nạp, sau `t` vòng ta đã lấy đúng `t` người cao nhất theo thứ tự giảm.
- **Cách `dict`:** hàm `h → who[h]` là song ánh trên tập chiều cao nên tra ngược không mất thông tin.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(names)`.

| Cách | Time | Space |
|---|---|---|
| 1. Selection sort thủ công | `O(n^2)` | `O(n)` |
| 2. `zip` + `sorted` | `O(n log n)` | `O(n)` |
| 3. Sắp xếp chỉ số | `O(n log n)` | `O(n)` |
| 4. `dict` + `sorted` | `O(n log n)` | `O(n)` |

Với `n <= 1000`, tất cả đều nhanh. Độ dài tên tối đa 20 nên so sánh/sao chép chuỗi coi là hằng số. Timsort: worst `O(n log n)`, tốt hơn (gần `O(n)`) nếu đầu vào đã gần có thứ tự.

## 12. Bộ test

Test 1: ví dụ của đề và các edge case. Đối chiếu tất cả cài đặt.

In [15]:
impls = [SolutionBrute(), SolutionZip(), SolutionIndex(), SolutionDict(), Solution()]
tests = [
    (["Mary", "John", "Emma"], [180, 165, 170], ["Mary", "Emma", "John"]),
    (["Alice", "Bob", "Bob"], [155, 185, 150], ["Bob", "Alice", "Bob"]),
    (["A"], [1], ["A"]),                                    # n = 1
    (["a", "B"], [1, 100000], ["B", "a"]),                  # biên giá trị
    (["x", "y", "z"], [1, 2, 3], ["z", "y", "x"]),          # tăng dần -> đảo ngược hoàn toàn
    (["x", "y", "z"], [3, 2, 1], ["x", "y", "z"]),          # đã đúng thứ tự
    (["Zed", "adam", "Bob"], [10, 30, 20], ["adam", "Bob", "Zed"]),   # tên không liên quan tới thứ tự
    (["Sam", "Sam", "Sam"], [5, 9, 7], ["Sam", "Sam", "Sam"]),         # toàn tên trùng
    (["Li", "Ann"], [70, 69], ["Li", "Ann"]),
]
for nm, hs, exp in tests:
    for im in impls:
        got = im.sortPeople(list(nm), list(hs))
        assert got == exp, (type(im).__name__, nm, hs, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 9


Test 2: ngẫu nhiên có seed, đúng constraints: `1 <= n <= 1000`, tên dài `1..20` gồm chữ cái hoa/thường (có thể trùng), chiều cao **đôi một khác nhau** trong `1..10^5`. Oracle: dựng danh sách `(chiều cao, chỉ số)` rồi sắp xếp giảm dần.

In [16]:
import random, string
random.seed(2418)
letters = string.ascii_letters

def rand_name():
    return ''.join(random.choice(letters) for _ in range(random.randint(1, 20)))

for t in range(300):
    n = random.randint(1, 60) if t < 280 else random.randint(900, 1000)
    pool = [rand_name() for _ in range(max(1, n // 3))] if t % 3 == 0 else None   # nhiều tên trùng
    nm = [random.choice(pool) if pool else rand_name() for _ in range(n)]
    hs = random.sample(range(1, 100001), n)                                          # đôi một khác nhau
    assert len(set(hs)) == n and all(1 <= len(x) <= 20 for x in nm)
    exp = [nm[i] for _, i in sorted(((h, i) for i, h in enumerate(hs)), reverse=True)]
    for im in impls:
        assert im.sortPeople(list(nm), list(hs)) == exp, (type(im).__name__, n)
print("300 test ngẫu nhiên OK (gồm 20 test cỡ ~1000)")

300 test ngẫu nhiên OK (gồm 20 test cỡ ~1000)


Test 3: kiểm tra tính chất — kết quả là hoán vị của `names`; đầu vào không bị sửa; và thứ tự chiều cao tương ứng giảm ngặt.

In [17]:
random.seed(5)
nm = [rand_name() for _ in range(50)]
hs = random.sample(range(1, 100001), 50)
copy_nm, copy_hs = list(nm), list(hs)
res = Solution().sortPeople(nm, hs)
assert sorted(res) == sorted(nm), "phải là hoán vị của names"
assert nm == copy_nm and hs == copy_hs, "không được sửa input"
back = {h: n for h, n in zip(hs, nm)}
assert [back[h] for h in sorted(hs, reverse=True)] == res
print("hoán vị / không sửa input / thứ tự: OK")

hoán vị / không sửa input / thứ tự: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sắp xếp **tăng dần** thay vì giảm dần

In [18]:
def wrong_ascending(names, heights):
    return [name for _, name in sorted(zip(heights, names))]        # SAI: thiếu reverse=True

nm, hs = ["Mary", "John", "Emma"], [180, 165, 170]
print("sai :", wrong_ascending(nm, hs))
print("đúng:", Solution().sortPeople(nm, hs))
assert wrong_ascending(nm, hs) != Solution().sortPeople(nm, hs)

sai : ['John', 'Emma', 'Mary']
đúng: ['Mary', 'Emma', 'John']


Vì sao sai: mặc định `sorted` tăng dần nên người thấp nhất đứng đầu; đề yêu cầu cao nhất đứng đầu. **Sửa:** thêm `reverse=True` (hoặc khoá `-heights[i]`).

### Lỗi 2: ghép cặp theo thứ tự `(tên, chiều cao)` → sắp theo **tên**

In [19]:
def wrong_by_name(names, heights):
    return [name for name, _ in sorted(zip(names, heights), reverse=True)]   # SAI: so sánh tên trước

print("sai :", wrong_by_name(nm, hs))
print("đúng:", Solution().sortPeople(nm, hs))
assert wrong_by_name(nm, hs) != Solution().sortPeople(nm, hs)

sai : ['Mary', 'John', 'Emma']
đúng: ['Mary', 'Emma', 'John']


Vì sao sai: khi sắp các tuple, Python so sánh phần tử **đầu** trước. Với `(tên, chiều cao)`, thứ tự do tên quyết định (`"Mary" > "John" > "Emma"` theo bảng chữ cái, giảm dần), hoàn toàn không liên quan chiều cao. **Sửa:** đặt chiều cao ở vị trí đầu hoặc dùng `key`.

### Lỗi 3: dùng `dict` tên → chiều cao (mất người khi trùng tên)

In [20]:
def wrong_dict_name(names, heights):
    h_of = dict(zip(names, heights))                  # SAI: tên trùng bị ghi đè
    return sorted(h_of, key=lambda n: h_of[n], reverse=True)

nm3, hs3 = ["Alice", "Bob", "Bob"], [155, 185, 150]
print("sai :", wrong_dict_name(nm3, hs3))
print("đúng:", Solution().sortPeople(nm3, hs3))
assert wrong_dict_name(nm3, hs3) != Solution().sortPeople(nm3, hs3)

sai : ['Alice', 'Bob']
đúng: ['Bob', 'Alice', 'Bob']


Vì sao sai: khoá `"Bob"` xuất hiện hai lần nên chỉ còn một mục (chiều cao 150), và kết quả chỉ có 2 phần tử thay vì 3. **Sửa:** dùng chiều cao làm khoá (đôi một khác nhau) hoặc sắp theo chỉ số.

### Lỗi 4 (bonus): gán kết quả của `list.sort()`

`names.sort(key=...)` sắp xếp **tại chỗ** và trả `None`. Viết `return names.sort(...)` sẽ trả về `None`. Dùng `sorted(...)` khi cần giá trị trả về, hoặc `sort` rồi `return names`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sắp xếp mảng A theo giá trị của mảng B" → `zip(B, A)` rồi `sorted`, hoặc **argsort** `sorted(range(n), key=B.__getitem__)`.
- Nhớ cặp là `(khoá, dữ liệu)`: khoá đứng **trước** thì tuple tự sắp theo khoá.
- Nếu khoá có thể **trùng**, hãy xác định quy tắc phá hoà (thêm khoá phụ); nếu đề đảm bảo khác nhau (như bài này) có thể dùng `dict` tra ngược.
- Giảm dần: `reverse=True` hoặc khoá âm `-h`. Với chuỗi dùng `reverse=True` (không thể đổi dấu).
- Với `n <= 1000`, đừng ngại dùng `sorted`; hiệu năng không phải vấn đề.
- Kiểm tra kết quả với ví dụ có **tên trùng** như `["Alice","Bob","Bob"]`.

## 15. Biến thể của bài

1. Chiều cao có thể **trùng**: khi bằng nhau thì sắp theo tên tăng dần (khoá `(-h, name)`).
2. Sắp xếp giảm dần theo chiều cao nhưng trả về **cả cặp** `(tên, chiều cao)`.
3. Sắp tăng dần theo **độ dài tên**, hoà thì theo chiều cao giảm dần.
4. Trả về chỉ **k người cao nhất** (dùng `heapq.nlargest`).
5. Sắp xếp mảng theo thứ tự tần suất giảm dần (Sort Characters By Frequency) — cùng ý "sắp theo khoá".

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Selection sort thủ công | tìm max lặp lại | `O(n^2)` | `O(n)` | học thuật toán sắp xếp, `n` rất nhỏ |
| 2. `zip` + `sorted` | cặp `(chiều cao, tên)` rồi sắp | `O(n log n)` | `O(n)` | **ngắn nhất**, dễ đọc |
| 3. Sắp xếp chỉ số | `sorted(range(n), key=...)` | `O(n log n)` | `O(n)` | nhiều mảng song song, cần `order` chung |
| 4. `dict` + `sorted` | tra tên theo chiều cao | `O(n log n)` | `O(n)` | khi khoá chắc chắn duy nhất |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `names = ["Tom", "Ann", "Joe"]`, `heights = [150, 190, 170]` → kết quả?

**Bài 2 (tính tay):** sắp xếp `["P", "Q", "R", "S"]` với `heights = [10, 40, 20, 30]` giảm dần; viết các cặp sau khi sắp.

**Bài 3 (điền TODO):** hoàn thành `sort_people_asc(names, heights)` sắp **tăng dần** (người thấp nhất đứng đầu).

**Bài 4 (tự viết lại):** viết `tallest_k(names, heights, k)` trả về `k` người cao nhất (đã theo thứ tự giảm dần).

**Bài 5 (mini challenge):** viết `sort_people_ties(names, heights)` trong trường hợp chiều cao **có thể trùng**: giảm dần theo chiều cao, nếu bằng nhau thì theo tên **tăng dần** (A→Z).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().sortPeople(["Tom", "Ann", "Joe"], [150, 190, 170]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def sort_people_asc(names, heights):
    # TODO: trả về names theo chiều cao TĂNG dần
    pass

['Ann', 'Joe', 'Tom']


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** luôn giữ cặp (chiều cao, tên) đi cùng nhau; chiều cao là khoá.

**Mức 2 (vừa):** Bài 3: bỏ `reverse=True`. Bài 4: sắp giảm dần rồi cắt `[:k]`. Bài 5: khoá là tuple `(-chiều cao, tên)` để giảm dần theo chiều cao và tăng dần theo tên.

**Mức 3 (gần đáp án):** Bài 3: `[n for _, n in sorted(zip(heights, names))]`. Bài 4: `Solution().sortPeople(names, heights)[:k]`. Bài 5: `[names[i] for i in sorted(range(len(names)), key=lambda i: (-heights[i], names[i]))]`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1
assert Solution().sortPeople(["Tom", "Ann", "Joe"], [150, 190, 170]) == ["Ann", "Joe", "Tom"]

# Bài 2: (40,Q) (30,S) (20,R) (10,P)
pairs = sorted(zip([10, 40, 20, 30], ["P", "Q", "R", "S"]), reverse=True)
assert pairs == [(40, "Q"), (30, "S"), (20, "R"), (10, "P")]

# Bài 3
def sort_people_asc(names, heights):
    return [n for _, n in sorted(zip(heights, names))]
assert sort_people_asc(["Mary", "John", "Emma"], [180, 165, 170]) == ["John", "Emma", "Mary"]

# Bài 4
def tallest_k(names, heights, k):
    return Solution().sortPeople(names, heights)[:k]
assert tallest_k(["Mary", "John", "Emma"], [180, 165, 170], 2) == ["Mary", "Emma"]
assert tallest_k(["Mary", "John", "Emma"], [180, 165, 170], 0) == []

# Bài 5
def sort_people_ties(names, heights):
    return [names[i] for i in sorted(range(len(names)), key=lambda i: (-heights[i], names[i]))]
assert sort_people_ties(["Bo", "Al", "Cy"], [170, 170, 180]) == ["Cy", "Al", "Bo"]
assert sort_people_ties(["b", "a"], [1, 1]) == ["a", "b"]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sắp mảng này theo khoá của mảng kia":**

```python
order = sorted(range(len(names)), key=lambda i: heights[i], reverse=True)
return [names[i] for i in order]
```

Hoặc ngắn hơn: `[n for _, n in sorted(zip(heights, names), reverse=True)]`.

**Checklist trước khi submit:**

- [ ] Khoá sắp xếp là **chiều cao** (không phải tên)?
- [ ] Đã có `reverse=True` (cao → thấp)?
- [ ] Có thể có tên trùng mà không bị mất (không dùng `dict` tên → chiều cao)?
- [ ] Không gán kết quả của `list.sort()` (trả `None`)?
- [ ] Đã thử `n = 1` và trường hợp tên trùng `["Alice","Bob","Bob"]`?
- [ ] Đã đối chiếu các cách bằng test ngẫu nhiên có seed (chiều cao đôi một khác nhau)?
- [ ] Chữ ký đúng `sortPeople(self, names: List[str], heights: List[int]) -> List[str]`?